# 🧠 AetherMind: AI Startup Mentor Fine-Tuning 🚀

Welcome to the **AetherMind** fine-tuning notebook. This notebook is designed to run on **Google Colab** (using a free T4 GPU) or any server with a GPU. It uses **Unsloth** to speed up the training of the **Gemma-2-2B-it** or **Gemma-2-7B-it** model by 2x and reduce memory usage by 80%!

### 📌 Prerequisites
- Ensure your runtime type is set to **GPU** (Runtime -> Change runtime type -> T4 GPU).
- No complex local setup required! We will load the dataset directly into the notebook.

---

## 🛠️ Step 1: Install Dependencies

We install `unsloth`, `trl`, `peft`, `accelerate`, and `bitsandbytes`.

In [ ]:
# Install Unsloth and essential libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets

## 📦 Step 2: Load the Model and Tokenizer

We will load the pre-quantized 4-bit model **Gemma 2 2B Instruct** or **Gemma 2 7B Instruct** from Unsloth. The 4-bit version uses very little memory, meaning it fits easily on a free T4 GPU (under 8GB of VRAM).

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Supports any value (RoPE Scaling handled automatically)
dtype = None          # None for auto detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True   # Use 4bit quantization to reduce memory usage

# Choose model:
# "unsloth/gemma-2-2b-it-bnb-4bit" - Super fast, lightweight (~1.6 GB)
# "unsloth/gemma-2-7b-it-bnb-4bit" - High quality, slightly slower (~5.5 GB)
model_name = "unsloth/gemma-2-2b-it-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    dtype = dtype,
    # token = "YOUR_HF_TOKEN" # Add your Hugging Face token here if using gated models
)

## 🎯 Step 3: Setup LoRA (Low-Rank Adaptation)

LoRA allows us to fine-tune only a fraction of the model parameters (usually <1%), making the process fast and resource-efficient while retaining the model's general capabilities.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank (Choose 8, 16, 32, 64, 128)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized for 0
    bias = "none",    # Optimized for "none"
    use_gradient_checkpointing = "unsloth", # Very low VRAM usage
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 📊 Step 4: Load and Prepare the Dataset

We load our high-quality **AetherMind** Indian startup advisory dataset. We will define the dataset inline below so this notebook can run completely independently without requiring file uploads!

In [ ]:
import json
from datasets import Dataset

# Direct inline loading of the AetherMind mentoring dataset
aethermind_data = [
  {
    "instruction": "I have an idea for a hyperlocal delivery service for organic vegetables in Tier 2 Indian cities. How do I validate this?",
    "input": "",
    "output": "To validate your hyperlocal organic veg delivery idea in Tier 2 cities, follow these steps: 1. **Identify the pain point**: Talk to 20-30 households to see if they actually care about 'organic' or if they just want 'fresh' and 'cheap'. 2. **MVP (Minimum Viable Product)**: Don't build an app yet. Start a WhatsApp group for a specific apartment complex or neighborhood. 3. **Manual Sourcing**: Buy the produce yourself from a local organic farmer or mandi and deliver it. 4. **Retention Check**: See if the same customers order again next week. In India, trust is bigger than tech—building that trust manually first is key."
  },
  {
    "instruction": "Explain the difference between a Private Limited company and an LLP for an Indian startup.",
    "input": "",
    "output": "A Private Limited (Pvt Ltd) company is the gold standard for startups looking to raise venture capital. It allows for equity dilution and ESOPs. An LLP (Limited Liability Partnership) is better for bootstrapped businesses or service agencies as it has fewer compliance requirements and no Dividend Distribution Tax, but it's very difficult to raise VC funding for an LLP because investors cannot easily hold shares."
  },
  {
    "instruction": "What should be included in a pre-seed investor pitch deck?",
    "input": "",
    "output": "A standard pre-seed deck should have 10-12 slides: 1. **Problem**: What specific pain are you solving? 2. **Solution**: How does your product fix it? 3. **Market Size (TAM/SAM/SOM)**: Use Indian market data (e.g., '100M smartphone users'). 4. **Business Model**: How will you make money? 5. **Traction**: Any early signs of interest? 6. **Team**: Why are you the right founders? 7. **The Ask**: How much money do you need and what will you do with it?"
  },
  {
    "instruction": "How can I apply for the Startup India DPIIT recognition?",
    "input": "",
    "output": "To get DPIIT recognition: 1. Register your business as a Pvt Ltd or LLP. 2. Go to the Startup India portal (startupindia.gov.in). 3. Fill the 'Recognition Application'. 4. Provide a write-up on how your startup is innovative or has scalability potential. 5. Once recognized, you can apply for tax exemptions (80-IAC) and easier public procurement norms."
  },
  {
    "instruction": "Write a cold email to a seed-stage VC in India.",
    "input": "My startup is 'QuickDukaan', a Shopify for rural artisans.",
    "output": "Subject: QuickDukaan: Solving e-commerce for 5M rural artisans\n\nHi [Investor Name],\n\nI'm [Your Name], founder of QuickDukaan. We've built a mobile-first platform that lets rural artisans set up online stores in 30 seconds via WhatsApp.\n\nIn the last 3 months, we've onboarded 500 artisans from Rajasthan and processed \u20b910L in GMV with zero marketing spend.\n\nI saw your investment in [Similar Portfolio Company] and thought our focus on the 'Next Billion Users' would interest you. Would you be open to a 15-minute chat next Tuesday?\n\nBest,\n[Your Name]"
  },
  {
    "instruction": "What are the common terms in an Indian Angel Network (IAN) term sheet?",
    "input": "",
    "output": "Common terms include: 1. **Valuation**: Pre-money vs Post-money. 2. **Liquidation Preference**: Usually 1x non-participating. 3. **Anti-Dilution**: Usually Broad-based Weighted Average. 4. **Board Seats**: Investors usually want one board observer or director seat. 5. **Founder Vesting**: Typically 4 years with a 1-year cliff. 6. **Rights of First Refusal (ROFR)**: Investors get first dibs on buying shares being sold by others."
  },
  {
    "instruction": "How do I handle a conflict with my co-founder regarding equity split?",
    "input": "",
    "output": "Equity split conflicts are best handled early. 1. **Value-based Split**: Use a framework like 'Founders' Pie' which weighs roles, ideas, and initial capital. 2. **Vesting is Mandatory**: Ensure all founders have a 4-year vesting schedule. This protects the company if someone leaves early. 3. **Transparent Conversation**: Sit down and list each person's long-term commitment. If it's not 50/50, explain why based on contribution, not just who 'came up with the idea'."
  },
  {
    "instruction": "Explain GST for a new software-as-a-service (SaaS) startup in India.",
    "input": "",
    "output": "For SaaS: 1. GST registration is mandatory if turnover exceeds \u20b920 Lakhs (or \u20b910 Lakhs in some states). 2. Export of services (selling to US/Europe) is 'Zero-rated' under GST, provided you file a Letter of Undertaking (LUT). 3. You must issue GST-compliant invoices. 4. Input Tax Credit (ITC) allows you to set off the GST you pay on expenses (like cloud servers or laptops) against the GST you collect from customers."
  },
  {
    "instruction": "What is the 'The Mom Test' approach to customer interviews?",
    "input": "",
    "output": "The Mom Test means talking to customers in a way that even your mom can't lie to you. 1. Talk about their life, not your idea. 2. Ask about specific things that happened in the past, not opinions about the future. 3. Talk less and listen more. Instead of 'Would you buy this?', ask 'Tell me about the last time you faced this problem and how you solved it.'"
  },
  {
    "instruction": "How do I set up an ESOP pool for my first 5 employees?",
    "input": "",
    "output": "1. **Reserve the Pool**: Usually 10% of equity is set aside for the ESOP pool. 2. **Define the Policy**: Set a vesting period (e.g., 4 years) and a cliff (1 year). 3. **Grant Letters**: Issue formal letters stating the number of options and exercise price. 4. **Exercise Price**: Keep it low (often face value like \u20b910) to make it a real benefit. Consult a CS to ensure the board resolution and filings are done correctly."
  }
]

# Create Dataset
dataset = Dataset.from_list(aethermind_data)

# Define Gemma 2 Prompt Template format
prompt_template = """<start_of_turn>user
{instruction}
{input}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn>"""

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(format_prompts, batched = True)
print("Dataset prepared! Total samples:", len(dataset))

## 🚀 Step 5: Configure and Run the Trainer

We use the **SFTTrainer** from Hugging Face's `trl` library to execute Supervised Fine-Tuning. Unsloth's optimized training engine makes this incredibly fast!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to True for much faster packing of shorter inputs
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this for higher accuracy (e.g. 100-200)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training completed successfully!")

## 🧪 Step 6: Test the Fine-Tuned Model

Let's test the model's new capabilities. We'll ask it a startup mentoring question that it was trained on to ensure it outputs in our custom expert style!

In [ ]:
# Enable fast inference mode with Unsloth
FastLanguageModel.for_inference(model)

test_prompt = "How do I validate my startup idea without building a complex product first?"

inputs = tokenizer(
    [f"<start_of_turn>user\n{test_prompt}<end_of_turn>\n<start_of_turn>model\n"],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:", test_prompt)
print("\nGenerated Response:")
print(response.split("model\n")[-1])

## 💾 Step 7: Export and Save the Model

You can save the trained model as a **LoRA Adapter** locally, or push it directly to **Hugging Face** to load it online later!

In [ ]:
# Save LoRA weights locally in Colab
model.save_pretrained("aethermind_gemma2_lora")
tokenizer.save_pretrained("aethermind_gemma2_lora")
print("Saved LoRA weights locally to 'aethermind_gemma2_lora'")

# Optional: Push model and tokenizer to Hugging Face
# You need to login to HF first using: from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your_username/aethermind_gemma2_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_username/aethermind_gemma2_lora", token = "YOUR_HF_TOKEN")